<h1 style="text-align: center;">Joined Data Exploration & EDA</h1>

This notebook combines the selected tables from the SQLite database into a match-level dataset for further exploration.

The initial focus is on understanding the joined data, checking its structure and quality, exploring feature distributions and relationships, and identifying potential issues that may affect the modelling process.

No final feature selection or modelling decisions are made at this stage. These decisions will be based on the findings from the EDA.

---

## Data Import

In [3]:
import pandas as pd
import sqlite3

In [4]:
connection = sqlite3.connect(r"../Data/database.sqlite")

In [5]:
match_df = pd.read_sql("SELECT * FROM Match", connection)
match_df.head()

,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
0,1,1,1,2008/2009,1,2008-08-17 00:00:00,492473,9987,9993,1,...,4.00,1.65,3.40,4.50,1.78,3.25,4.00,1.73,3.40,4.20
1,2,1,1,2008/2009,1,2008-08-16 00:00:00,492474,10000,9994,0,...,3.80,2.00,3.25,3.25,1.85,3.25,3.75,1.91,3.25,3.60
2,3,1,1,2008/2009,1,2008-08-16 00:00:00,492475,9984,8635,0,...,2.50,2.35,3.25,2.65,2.50,3.20,2.50,2.30,3.20,2.75
3,4,1,1,2008/2009,1,2008-08-17 00:00:00,492476,9991,9998,5,...,7.50,1.45,3.75,6.50,1.50,3.75,5.50,1.44,3.75,6.50
4,5,1,1,2008/2009,1,2008-08-16 00:00:00,492477,7947,9985,1,...,1.73,4.50,3.40,1.65,4.50,3.50,1.65,4.75,3.30,1.67


In [6]:
league_df = pd.read_sql("SELECT * FROM League", connection)
league_df.head()

,id,country_id,name
0,1,1,Belgium Jupiler League
1,1729,1729,England Premier League
2,4769,4769,France Ligue 1
3,7809,7809,Germany 1. Bundesliga
4,10257,10257,Italy Serie A


In [7]:
team_df = pd.read_sql("SELECT * FROM Team", connection)
team_df.head()

,id,team_api_id,team_fifa_api_id,team_long_name,team_short_name
0,1,9987,673.0,KRC Genk,GEN
1,2,9993,675.0,Beerschot AC,BAC
2,3,10000,15005.0,SV Zulte-Waregem,ZUL
3,4,9994,2007.0,Sporting Lokeren,LOK
4,5,9984,1750.0,KSV Cercle Brugge,CEB


In [8]:
team_attributes_df=pd.read_sql("SELECT * FROM Team_Attributes", connection)
team_attributes_df.head()

,id,team_fifa_api_id,team_api_id,date,buildUpPlaySpeed,buildUpPlaySpeedClass,buildUpPlayDribbling,buildUpPlayDribblingClass,buildUpPlayPassing,buildUpPlayPassingClass,...,chanceCreationShooting,chanceCreationShootingClass,chanceCreationPositioningClass,defencePressure,defencePressureClass,defenceAggression,defenceAggressionClass,defenceTeamWidth,defenceTeamWidthClass,defenceDefenderLineClass
0,1,434,9930,2010-02-22 00:00:00,60,Balanced,NaN,Little,50,Mixed,...,55,Normal,Organised,50,Medium,55,Press,45,Normal,Cover
1,2,434,9930,2014-09-19 00:00:00,52,Balanced,48.0,Normal,56,Mixed,...,64,Normal,Organised,47,Medium,44,Press,54,Normal,Cover
2,3,434,9930,2015-09-10 00:00:00,47,Balanced,41.0,Normal,54,Mixed,...,64,Normal,Organised,47,Medium,44,Press,54,Normal,Cover
3,4,77,8485,2010-02-22 00:00:00,70,Fast,NaN,Little,70,Long,...,70,Lots,Organised,60,Medium,70,Double,70,Wide,Cover
4,5,77,8485,2011-02-22 00:00:00,47,Balanced,NaN,Little,52,Mixed,...,52,Normal,Organised,47,Medium,47,Press,52,Normal,Cover


---

## Data Merge

In [9]:
match_df["year"] = pd.to_datetime(match_df["date"]).dt.year

In [10]:
data = match_df.merge(league_df, left_on="league_id", right_on="id", how="left", validate="m:1")

In [11]:
team_attributes_df["year"] = pd.to_datetime(
    team_attributes_df["date"]
).dt.year

In [12]:
master_team_df = team_attributes_df.merge(team_df, on="team_api_id", how="left", validate="m:1")

In [13]:
master_team_df.drop(columns=['id_y','team_fifa_api_id_y'], inplace=True)

In [14]:
master_team_df.rename(columns={'id_x':'id','team_fifa_api_id_x':'team_fifa_api_id'}, inplace=True)

In [19]:
home_team_data = master_team_df.copy()
away_team_data = master_team_df.copy()

In [20]:
home_team_data = home_team_data.add_prefix("home_")
away_team_data = away_team_data.add_prefix("away_")

In [21]:
data = data.merge(home_team_data, left_on=["home_team_api_id","year"], right_on=["home_team_api_id", "home_year"], how='left', validate="m:m")

In [22]:
data = data.merge(away_team_data, left_on=["away_team_api_id","year"], right_on=["away_team_api_id","away_year"], how='left', validate="m:m")

In [23]:
data_profile = pd.DataFrame({
    "columns":data.columns,
    "data_types":data.dtypes.values,
    "missing_values":data.isna().sum().values,
    "missing_percentage":(data.isna().mean()*100).values,
    "unique_values":data.nunique().values
})

In [24]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
data_profile

,columns,data_types,missing_values,missing_percentage,unique_values
0,id_x,int64,0,0.000000,25979
1,country_id_x,int64,0,0.000000,11
2,league_id,int64,0,0.000000,11
3,season,str,0,0.000000,8
4,stage,int64,0,0.000000,38
5,date,str,0,0.000000,1694
6,match_api_id,int64,0,0.000000,25979
7,home_team_api_id,int64,0,0.000000,299
8,away_team_api_id,int64,0,0.000000,299
9,home_team_goal,int64,0,0.000000,11


In [29]:
data.to_parquet("../Data/joined_match_data_raw.parquet", index=False)
data_profile.to_csv("../Data/joined_data_profile_raw.csv", index=False)

In [25]:
data.shape

(25979, 173)

---

## Creating the Match Outcome Target

The target `FTR` represents the final outcome of each match based on the
home and away team goals. It classifies each match as a home win (`H`),
draw (`D`), or away win (`A`).

In [31]:
import numpy as np

In [32]:
data['FTR'] = np.select([ data["home_team_goal"] > data["away_team_goal"], data["home_team_goal"] < data["away_team_goal"]],["H","A"], default="D")

#### Target Exploration

For this stage, FTR will be used as our main target variable during exploration and EDA. The existing columns, including the goal columns used to create FTR, will not be removed yet. Column removal and other dataset changes will be carried out later after the exploratory analysis and feature-selection process.

---

## Initial Column Classification

At this stage, we are separating columns that can be removed with high
confidence from those that require further investigation. **No feature
selection based on predictive usefulness is being performed yet.**

### Columns to Remove with High Confidence

The following columns are considered redundant, technical identifiers,
constant features, or direct target leakage:

- Technical/identifier columns: `id_x`, `country_id_x`, `league_id`,
  `match_api_id`, `home_team_api_id`, `away_team_api_id`, `id_y`,
  `country_id_y`, `home_id`, `home_team_fifa_api_id`, `away_id`,
  `away_team_fifa_api_id`
- Redundant date information: `date`, `home_date`, `away_date`,
  `home_year`, `away_year`
  - The main `year` column will be retained.
- Target-leaking columns: `home_team_goal`, `away_team_goal`
- Constant column: `away_player_Y2`

### Columns to Investigate Further

The following will **not be removed yet**. Their usefulness will be
examined during EDA:

- Match statistics such as `goal` through `possession`, which have
  substantial and synchronized missing values.
- Betting-odds columns with high missingness.
- `buildUpPlayDribbling` and `away_buildUpPlayDribbling`, because of their
  high missing percentages.
- Other high-missingness team-attribute features.

The purpose of this stage is therefore to remove only what we are already
confident is unnecessary, while keeping potentially useful features for
further EDA and evidence-based selection.

---